<center>
  <h1><b>SeedUp - Smart Torrent Management V1</b></h1>

---


### **Important Notes:**

1. **Free Tier Limitations:** Google Colab free tier has:
   - Limited runtime
   - Limited disk space (~100GB)
   - Session may disconnect if idle

2. **Legal Usage Only:** Only download content you have the right to download.

3. **VikingFile Storage:** VikingFile offers unlimited free storage per file.

---
# **📥 Step 1: Get Project Files**

Clone the torrent downloader and VikingFile uploader scripts from your GitHub repo.

This clones your project files from your private GitHub repo instead of writing them out by hand each time.

**One-time setup:** In Colab, click the key icon (🔑) in the left sidebar → **Add new secret** → name it `GITHUB_TOKEN` → paste your GitHub Personal Access Token (repo read scope) as the value → toggle **Notebook access** on.

Repo: `https://github.com/MahdiAfsar76/Torrent-vikingfile`

In [ ]:
#@title **Clone Project Files from GitHub** { display-mode: "form" }
import os
import stat
from google.colab import userdata

REPO_URL = "https://github.com/MahdiAfsar76/Torrent-vikingfile.git"  #@param {type:"string"}
REPO_NAME = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")

try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None

if os.path.exists(REPO_NAME):
    !rm -rf {REPO_NAME}

if token:
    # Use a GIT_ASKPASS helper script so the token never appears in the
    # clone URL, the printed command, or any error output.
    askpass_path = "/content/.git_askpass.sh"
    with open(askpass_path, "w") as f:
        f.write("#!/bin/sh\necho \"$GIT_TOKEN\"\n")
    os.chmod(askpass_path, stat.S_IRWXU)

    os.environ["GIT_ASKPASS"] = askpass_path
    os.environ["GIT_TOKEN"] = token
    os.environ["GIT_TERMINAL_PROMPT"] = "0"

    auth_repo_url = REPO_URL.replace("https://", "https://x-access-token@")
    !git clone {auth_repo_url}
else:
    print("⚠️ No GITHUB_TOKEN secret found — attempting an unauthenticated clone.")
    print("   This only works if the repo is public.")
    !git clone {REPO_URL}

%cd {REPO_NAME}


In [ ]:
#@title **Verify Project Files** { display-mode: "form" }
import os

required_files = [
    "main.py",
    "config.py",
    "vikingfile_uploader.py",
    "torrent_downloader.py",
    "requirements.txt",
]

missing = [f for f in required_files if not os.path.exists(f)]

if missing:
    print("⚠️ Missing files:", ", ".join(missing))
    print("Check that your repo contains these files and that the clone succeeded above.")
else:
    print("✅ All project files found!")


---
# **🔧 Step 2: Install Dependencies**

This will install all required packages including libtorrent and the VikingFile upload dependencies.

In [ ]:
#@title **Install Required Packages** { display-mode: "form" }
#@markdown Click the ▶️ button to install dependencies (takes ~2 minutes)

print("📦 Installing dependencies...\n")

# Install libtorrent
!pip install -r requirements.txt > /dev/null 2>&1


print("\n✅ All dependencies installed successfully!")
print("📌 Ready to proceed to the next step.")

In [ ]:
#@title **Verify Installations** { display-mode: "form" }
#@markdown Click the ▶️ button to verify all installations are working properly

import sys
import subprocess

def verify_installation():
    print("🔍 Verifying installations...\n")

    # Check Python version
    py_version = sys.version.split()[0]
    print(f"📌 Python Version: {py_version}")

    # Check libtorrent
    try:
        import libtorrent as lt
        print(f"📌 libtorrent Version: {lt.version}")
    except ImportError:
        print("❌ libtorrent not found!")
        return False

    # Check VikingFile upload dependencies
    try:
        import requests
        print("📌 requests library: ✓")
    except ImportError:
        print("❌ requests library not found!")
        return False

    print("\n✅ All dependencies verified successfully!")
    return True

verify_installation()

# **🔐 Step 3: Configure VikingFile**

VikingFile uploads use a simple account "user hash" instead of an OAuth login popup.

In [ ]:
#@title **Configure VikingFile Account (Optional)** { display-mode: "form" }
#@markdown VikingFile doesn't use a login popup like Google Drive. Instead, set
#@markdown your account's user hash directly in `config.py`
#@markdown (`VIKINGFILE_USER_HASH = "..."`) so uploads are associated with your
#@markdown account. Leave it blank to upload anonymously.

with open("config.py") as f:
    contents = f.read()

import re
match = re.search(r'VIKINGFILE_USER_HASH\s*=\s*"([^"]*)"', contents)
current_hash = match.group(1) if match else "(not set)"

print(f"📌 Current configured VikingFile user hash: {current_hash}")
print("📌 Edit config.py directly to change it.")


# **🗂️ Step 4: List Torrent Files (Optional)**

Preview which files are inside a torrent, without downloading any data. Use the printed index numbers to only download specific files in the next step (useful for multi-file torrents where you don't want everything).

In [ ]:
#@title **List Torrent Files** { display-mode: "form" }

#@markdown Paste magnet link OR path to .torrent file:
TORRENT_SOURCE_PREVIEW = "" #@param {type:"string"}

if not TORRENT_SOURCE_PREVIEW:
    print("⚠️ Please enter a magnet link or torrent file path!")
else:
    !python main.py files -t "{TORRENT_SOURCE_PREVIEW}"


# **🌐 Step 5: Download Torrent**

Download a torrent using either:
- **Magnet Link:** `magnet:?xt=urn:btih:...`
- **Torrent File:** Upload a .torrent file to Colab first

### Options:
- **Auto Upload:** Automatically upload to VikingFile after download
- **Skip Existing:** Skip files that already exist remotely

**Note:** Files will be uploaded to a folder named after the download automatically (or your custom folder path if specified)

In [ ]:
#@title **Download Torrent** { display-mode: "form" }

#@markdown ### 🔗 Enter Torrent Information
#@markdown Paste magnet link OR path to .torrent file:
TORRENT_SOURCE = "" #@param {type:"string"}

#@markdown ### 🗂️ File Selection (optional)
#@markdown File indices from the "List Torrent Files" step above, e.g. "0,2,5"
#@markdown or with ranges "0-3,7". Leave blank to download every file in the torrent.
SELECT_FILES = "" #@param {type:"string"}

#@markdown ### ⚙️ Options
AUTO_UPLOAD = True #@param {type:"boolean"}
SKIP_EXISTING = True #@param {type:"boolean"}
#@markdown Destination folder path on VikingFile (optional, e.g. "SeedUp/Movies"):
VIKINGFILE_PATH = "" #@param {type:"string"}

import sys

if not TORRENT_SOURCE:
    print("⚠️ Please enter a magnet link or torrent file path!")
else:
    print("🚀 Starting torrent download...\n")

    if SELECT_FILES:
        print(f"🗂️ Only downloading selected file indices: {SELECT_FILES}\n")

    if AUTO_UPLOAD:
        dest = VIKINGFILE_PATH if VIKINGFILE_PATH else "(auto folder name)"
        print(f"📁 Files will be uploaded to VikingFile: {dest}\n")

    # Build command
    cmd = f'python main.py download -t "{TORRENT_SOURCE}"'

    if SELECT_FILES:
        cmd += f' --select-files {SELECT_FILES}'

    if AUTO_UPLOAD:
        cmd += " --upload"
        if VIKINGFILE_PATH:
            cmd += f' -p "{VIKINGFILE_PATH}"'

    if not SKIP_EXISTING:
        cmd += " --no-skip"

    # Execute download
    !{cmd}


# **📤 Step 6: Upload Existing Files**

If you already have downloaded files and want to upload them to VikingFile separately.

**Note:** Files will be uploaded to a folder named after the download automatically (or your custom folder path if specified)

In [ ]:
#@title **Upload Files to VikingFile** { display-mode: "form" }

#@markdown ### 📁 File/Folder to Upload
LOCAL_PATH = "" #@param {type:"string"}

#@markdown ### ⚙️ Options
SKIP_EXISTING_FILES = True #@param {type:"boolean"}
#@markdown Destination folder path on VikingFile (optional, e.g. "SeedUp/Movies"):
VIKINGFILE_PATH = "" #@param {type:"string"}

import os

if not LOCAL_PATH:
    print("⚠️ Please enter a file or folder path to upload!")
elif not os.path.exists(LOCAL_PATH):
    print(f"⚠️ Path does not exist: {LOCAL_PATH}")
else:
    print("📤 Starting upload to VikingFile...\n")

    # Build command
    cmd = f'python main.py upload -p "{LOCAL_PATH}"'

    if VIKINGFILE_PATH:
        cmd += f' -r "{VIKINGFILE_PATH}"'

    if not SKIP_EXISTING_FILES:
        cmd += " --no-skip"

    # Execute upload
    !{cmd}


# **🔍 Step 7: Check Download Status**

Check if there's a paused download that can be resumed.

In [ ]:
#@title **Check Status** { display-mode: "form" }

!python main.py status

# **🧹 Step 8: Clear Session (Optional)**

Clear any saved download session if you want to start fresh.

In [ ]:
#@title **Clear Download Session** { display-mode: "form" }

!python main.py clear


---

## 📚 **Additional Information**

### Troubleshooting

**Download is slow:**
- This depends on the number of seeders and your torrent health
- Colab's network speed varies
- Try using different trackers or magnet links

**Upload failing:**
- Check your internet connection is stable
- Double-check your user hash in config.py if uploads aren't appearing in your account
- Check if files already exist remotely when using the skip option
- Destination folders on VikingFile are created automatically as part of the upload path

**Session disconnected:**
- Downloads are saved and can be resumed
- Use the status check to verify saved sessions
- Run the download command again with the same torrent
- Clear stuck sessions if needed

### Best Practices

1. **For large downloads:**
   - Monitor the session and be ready to resume if it disconnects
   - Use the keep-alive cell to prevent idle disconnects
   - Split very large downloads into smaller parts

2. **Storage management:**
   - All files are organized in the 'SeedUp Downloads' folder automatically
   - Clear the downloads folder after successful upload
   - Monitor Colab's disk space usage
   - Use `skip-existing` to avoid duplicate uploads

3. **Finding your uploads:**
   - Each uploaded file's `https://vikingfile.com/f/<hash>` link is printed after each upload
   - If you configured a user hash, uploads also appear in your VikingFile account's file list

### Session Management

The project uses a sophisticated session management system that:
- Saves download progress automatically
- Enables resuming from the exact point of interruption
- Maintains tracker and peer information
- Cleans up automatically after successful completion

---
<br>

### 🔗 **Links**

- [GitHub Repository](https://github.com/codercyco/SeedUp)
- [Report Issues](https://github.com/codercyco/SeedUp/issues)

<br>

---

<p><b>Liked it? You can buy me a coffee ☕</b></p>
<a href="https://www.buymeacoffee.com/codercyco" target="_blank">
  <img src="https://cdn.buymeacoffee.com/buttons/v2/default-blue.png" alt="Buy Me A Coffee" width="150">
</a>

<br>

---

<center>
<p>Made with ❤️ for the community</p>
<p>Created by <b>Ishara Deshapriya</b></p>

---